# G1 Academy Bonus - Task 8: high-level arm gestures, teach/repeat, and low-level pose interpolation

## Introduction
This task covers `notes.txt` section 6 in full: native high-level arm gestures through `G1ArmActionClient`, then a controller-ownership-safe path onto `rt/arm_sdk` (`release_arms`/`engage_arms`), and finally two general-purpose low-level pose helpers - `save_current_ll_pose` / `interpolate_to_ll_pose` - built up into `teach`/`repeat` for recording and replaying new multi-waypoint arm sequences.

In [ ]:
import threading
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

import sys
if ".." not in sys.path:
    sys.path.append("..")
from sdk_wrapper import ensure_channel_factory

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - High-level gestures via `G1ArmActionClient`
`G1ArmActionClient.ExecuteAction(action_id)` triggers a documented, firmware-baked gesture. These must never run concurrently with a low-level `rt/arm_sdk` command stream - finish or release one before starting the other.

In [ ]:
from unitree_sdk2py.g1.arm.g1_arm_action_client import G1ArmActionClient

HL_ARM_ACTIONS = {"release arm": 99, "clap": 17, "face wave": 25, "high wave": 26, "shake hand": 27, "hug": 19}

arm_action_client = G1ArmActionClient(); arm_action_client.SetTimeout(10.0); arm_action_client.Init()

def exec_arm_action(name, release_after_s=None):
    code = int(arm_action_client.ExecuteAction(HL_ARM_ACTIONS[name]))
    if release_after_s is not None:
        time.sleep(release_after_s)
        return int(arm_action_client.ExecuteAction(HL_ARM_ACTIONS["release arm"]))
    return code

def clap():
    return exec_arm_action("clap")

def face_wave():
    return exec_arm_action("face wave")

In [ ]:
#clap()
face_wave()

## Task 2 - Upper-body pose read + `rt/arm_sdk` writer
Every low-level helper below reads the current upper-body pose from `rt/lowstate` and writes a complete `LowCmd_` frame to `rt/arm_sdk`, including the weight byte at `motor_cmd[29]` that arbitrates between the default controller and this publisher.

In [ ]:
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_
from unitree_sdk2py.utils.crc import CRC

WAIST_JOINTS = (12, 13, 14)
LEFT_ARM_JOINTS = list(range(15, 22))
RIGHT_ARM_JOINTS = list(range(22, 29))
ARM_JOINTS = LEFT_ARM_JOINTS + RIGHT_ARM_JOINTS
UPPER_BODY_JOINTS = list(WAIST_JOINTS) + ARM_JOINTS
_crc = CRC()
arm_sdk_pub = ChannelPublisher("rt/arm_sdk", LowCmd_); arm_sdk_pub.Init()

def current_upper_body_pose(timeout_s=3.0):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if lowstate_sub.message is not None:
            return {j: float(lowstate_sub.message.motor_state[j].q) for j in UPPER_BODY_JOINTS}
        time.sleep(0.02)
    raise TimeoutError("No fresh rt/lowstate.")

def write_arm_sdk_pose(targets, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0):
    msg = unitree_hg_msg_dds__LowCmd_(); msg.mode_pr = 0; msg.mode_machine = 0
    msg.motor_cmd[29].q = max(0.0, min(1.0, float(weight)))
    for joint, q in targets.items():
        cmd = msg.motor_cmd[int(joint)]
        cmd.mode = 1; cmd.q = float(q); cmd.dq = 0.0; cmd.tau = 0.0
        cmd.kp = waist_kp if int(joint) in WAIST_JOINTS else kp
        cmd.kd = waist_kd if int(joint) in WAIST_JOINTS else kd
    msg.crc = _crc.Crc(msg)
    arm_sdk_pub.Write(msg)

_zero_stiffness_state = {"thread": None, "stop": None, "arm": None, "frames": 0}

def _stop_zero_stiffness_stream():
    state = _zero_stiffness_state
    if state["stop"] is not None:
        state["stop"].set()
        state["thread"].join(timeout=2.0)
    result = {"arm": state["arm"], "frames": state["frames"]}
    state.update(thread=None, stop=None, arm=None, frames=0)
    return result

def arms_restore_stiffness():
    """Stop free-move mode and hold the current upper-body pose normally."""
    result = _stop_zero_stiffness_stream()
    pose = current_upper_body_pose()
    write_arm_sdk_pose(pose, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0)
    result["final_pose"] = pose
    return result

def arms_zero_stiffness(rate_hz=50.0, arm="both", handoff=True):
    """Keep one or both arms backdrivable until arms_restore_stiffness().

    This follows Robot.teach: arms receive zero kp/kd/tau, but the waist
    stays at its normal hold gains to protect robot stability.
    """
    state = _zero_stiffness_state
    arm = str(arm).lower()
    joints = {"left": LEFT_ARM_JOINTS, "right": RIGHT_ARM_JOINTS, "both": ARM_JOINTS}.get(arm)
    if joints is None:
        raise ValueError("arm must be 'left', 'right', or 'both'")
    if state["thread"] is not None and state["thread"].is_alive():
        if state["arm"] != arm:
            raise RuntimeError("Free-move mode is already active for a different arm selection")
        return {"active": True, "arm": arm, "frames": state["frames"]}
    if handoff:
        release_arms()
        engage_arms()
    waist_hold = current_upper_body_pose()
    stop = threading.Event()
    interval_s = 1.0 / max(1.0, float(rate_hz))
    state.update(stop=stop, arm=arm, frames=0)
    def worker():
        while not stop.is_set():
            pose = current_upper_body_pose()
            targets = {joint: waist_hold[joint] for joint in WAIST_JOINTS}
            targets.update({joint: pose[joint] for joint in joints})
            write_arm_sdk_pose(targets, weight=1.0, kp=0.0, kd=0.0, waist_kp=480.0, waist_kd=12.0)
            state["frames"] += 1
            stop.wait(interval_s)
    state["thread"] = threading.Thread(target=worker, name="arms-zero-stiffness", daemon=True)
    state["thread"].start()
    return {"active": True, "arm": arm, "joints": joints, "kp": 0.0, "kd": 0.0, "waist_kp": 480.0}

## Task 3 - `release_arms` / `engage_arms`: safe controller handoff
Ramp the `arm_sdk` weight smoothly (an ease curve, not a step) from 1 to 0 to hand control back to the default controller, or 0 to 1 to take it - this is what every low-level helper below must run before/after it owns `rt/arm_sdk`.

In [ ]:
def release_arms(steps=150, rate_hz=50.0):
    # A zero-stiffness worker must never publish during this ownership ramp.
    _stop_zero_stiffness_stream()
    pose = current_upper_body_pose()
    for i in range(steps + 1):
        ratio = i / steps
        fade = ratio * ratio * (3 - 2 * ratio)
        weight = 1.0 - fade
        write_arm_sdk_pose(pose, weight=weight, kp=30.0 * weight, kd=1.5 * weight, waist_kp=480.0 * weight, waist_kd=12.0 * weight)
        time.sleep(1.0 / rate_hz)
    return {"final_weight": 0.0}

def engage_arms(steps=50, rate_hz=50.0):
    pose = current_upper_body_pose()
    for i in range(steps + 1):
        weight = i / steps
        write_arm_sdk_pose(pose, weight=weight)
        time.sleep(1.0 / rate_hz)
    return {"final_weight": 1.0}

In [ ]:
engage_arms()

In [ ]:
release_arms()

## Task 4 - `save_current_ll_pose(name)` / `interpolate_to_ll_pose(name_or_pose, ...)`
Two general-purpose helpers: capture the current pose under a name (persisted to JSON so it survives a kernel restart), and smoothly interpolate from wherever the arm currently is to a saved (or literal) pose using the same ease curve as `release_arms`/`engage_arms`.

In [ ]:
import json
from pathlib import Path

_ll_pose_store_path = Path("ll_poses.json")
def _load_ll_poses():
    return json.loads(_ll_pose_store_path.read_text()) if _ll_pose_store_path.exists() else {}
def _save_ll_poses(poses):
    _ll_pose_store_path.write_text(json.dumps(poses, indent=2))
_ll_poses = _load_ll_poses()

def save_current_ll_pose(name):
    pose = current_upper_body_pose()
    _ll_poses[str(name)] = pose
    _save_ll_poses(_ll_poses)
    return pose

def interpolate_to_ll_pose(name_or_pose, duration_s=4.0, steps=150):
    target = _ll_poses[name_or_pose] if isinstance(name_or_pose, str) else name_or_pose
    target = {int(j): float(q) for j, q in target.items()}
    start = current_upper_body_pose()
    for step in range(1, steps + 1):
        ratio = step / steps
        smooth = ratio * ratio * (3 - 2 * ratio)
        frame = {j: start[j] + (target[j] - start[j]) * smooth for j in target}
        write_arm_sdk_pose(frame)
        time.sleep(duration_s / steps)
    return {"target": target, "steps": steps}

In [ ]:
arms_zero_stiffness()

In [ ]:
save_current_ll_pose("extended_right")

In [ ]:
interpolate_to_ll_pose("extended_right", duration_s=4.0)

## Task 5 - `teach(sequence_name)` / `repeat(sequence_name)`: recording new sequences
`teach` appends the current pose as the next waypoint of a named, persisted sequence - call it repeatedly (moving the arm by hand in damp mode, or via `interpolate_to_ll_pose` to intermediate poses, between calls) to build up a multi-waypoint motion. `repeat` plays every captured waypoint back in order using `interpolate_to_ll_pose`, so playback is exactly as smooth as any other low-level move here.

In [ ]:
_sequences_path = Path("arm_sequences.json")
def _load_sequences():
    return json.loads(_sequences_path.read_text()) if _sequences_path.exists() else {}
def _save_sequences(seqs):
    _sequences_path.write_text(json.dumps(seqs, indent=2))
_sequences = _load_sequences()

def _arm_joints(arm):
    joints = {"left": LEFT_ARM_JOINTS, "right": RIGHT_ARM_JOINTS, "both": ARM_JOINTS}.get(str(arm).lower())
    if joints is None:
        raise ValueError("arm must be 'left', 'right', or 'both'")
    return joints

def teach(sequence_name, reset=False, arm="both", rate_hz=50.0):
    """Record until Enter is pressed, while the selected arms stay backdrivable."""
    arm = str(arm).lower(); joints = _arm_joints(arm)
    arms_zero_stiffness(arm=arm)
    done = threading.Event()
    def wait_for_enter():
        try:
            input("Move the arm, then press Enter to finish recording... " )
        except EOFError:
            pass
        done.set()
    threading.Thread(target=wait_for_enter, name="teach-enter", daemon=True).start()
    interval_s = 1.0 / max(1.0, float(rate_hz))
    start = time.monotonic()
    timestamps, frames = [], []
    while not done.is_set():
        elapsed = time.monotonic() - start
        pose = current_upper_body_pose()
        timestamps.append(elapsed); frames.append(pose)
        done.wait(interval_s)
    if not frames:
        raise RuntimeError("No teach frames were captured.")
    _sequences[sequence_name] = {"format": "trajectory_v1", "arm": arm, "joints": joints, "timestamps": timestamps, "frames": frames}
    _save_sequences(_sequences)
    duration_s = timestamps[-1]
    release_arms()
    return {"sequence": sequence_name, "frames": len(frames), "duration_s": duration_s, "arm": arm, "released_to_ai": True}

def repeat(sequence_name, speed=1.0, rate_hz=50.0, start_ramp_s=0.8, final_hold_s=0.8, max_joint_speed=0.45):
    """Replay a recorded trajectory with a safe ramp, speed limit, final hold, and release."""
    sequence = _sequences[sequence_name]
    if not isinstance(sequence, dict) or sequence.get("format") != "trajectory_v1":
        raise ValueError("Re-record this sequence with teach(); legacy waypoint sequences are not safe to replay.")
    arm = sequence["arm"]; joints = [int(j) for j in sequence["joints"]]
    timestamps = [float(t) for t in sequence["timestamps"]]
    frames = [{int(j): float(q) for j, q in frame.items()} for frame in sequence["frames"]]
    if not frames or len(frames) != len(timestamps):
        raise ValueError("Recorded sequence has invalid frames or timestamps.")
    speed = max(1e-3, float(speed)); rate_hz = max(1.0, float(rate_hz)); dt = 1.0 / rate_hz
    release_arms(); engage_arms()
    try:
        start_pose = current_upper_body_pose(); waist_hold = {j: start_pose[j] for j in WAIST_JOINTS}; first = frames[0]
        max_delta = max(abs(first[j] - start_pose[j]) for j in joints)
        ramp_s = max(float(start_ramp_s), max_delta / max(1e-3, float(max_joint_speed)))
        steps = max(1, int(ramp_s * rate_hz))
        for step in range(1, steps + 1):
            smooth = (step / steps) ** 2 * (3.0 - 2.0 * step / steps)
            target = dict(start_pose); target.update({j: start_pose[j] + (first[j] - start_pose[j]) * smooth for j in joints})
            write_arm_sdk_pose(target, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0)
            time.sleep(dt)
        started = time.monotonic(); previous = {j: first[j] for j in joints}; index = 0
        while True:
            elapsed = (time.monotonic() - started) * speed
            if elapsed >= timestamps[-1]:
                break
            while index + 1 < len(timestamps) and timestamps[index + 1] <= elapsed:
                index += 1
            next_index = min(index + 1, len(timestamps) - 1)
            span = max(1e-6, timestamps[next_index] - timestamps[index])
            alpha = 0.0 if next_index == index else (elapsed - timestamps[index]) / span
            desired = {j: frames[index][j] + (frames[next_index][j] - frames[index][j]) * alpha for j in joints}
            target = current_upper_body_pose(); target.update(waist_hold)
            max_step = float(max_joint_speed) * dt
            for j in joints:
                previous[j] += max(-max_step, min(max_step, desired[j] - previous[j]))
                target[j] = previous[j]
            write_arm_sdk_pose(target, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0)
            time.sleep(dt)
        final = frames[-1]; deadline = time.monotonic() + max(0.0, float(final_hold_s))
        while time.monotonic() < deadline:
            target = current_upper_body_pose(); target.update(waist_hold); target.update({j: final[j] for j in joints})
            write_arm_sdk_pose(target, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0)
            time.sleep(dt)
    finally:
        release_arms()
    return {"sequence": sequence_name, "frames": len(frames), "duration_s": timestamps[-1] / speed, "arm": arm}

In [ ]:
teach("wave_sequence", reset=True, arm="right")  # records until Enter, then releases to AI

In [ ]:
repeat("wave_sequence")  # safe ramp, replay, final hold, then release to AI

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.